##### Init



In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS dbr_dev.yanquiel")
spark.sql(f"CREATE VOLUME IF NOT EXISTS dbr_dev.yanquiel.raw_data")

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.types import StructType, StringType, IntegerType, DoubleType
import re
import requests
import pandas as pd

catalog = "dbr_dev"
schema = "yanquiel"
volume_path = f"/Volumes/{catalog}/{schema}/raw_data/world_population.csv"

##### Bronze layer


In [0]:
population_schema = (
    StructType()
    .add("Rank", IntegerType())
    .add("CCA3", StringType())
    .add("Country/Territory", StringType())
    .add("Capital", StringType())
    .add("Continent", StringType())
    .add("2022 Population", IntegerType())
    .add("2020 Population", IntegerType())
    .add("2015 Population", IntegerType())
    .add("2010 Population", IntegerType())
    .add("2000 Population", IntegerType())
    .add("1990 Population", IntegerType())
    .add("1980 Population", IntegerType())
    .add("1970 Population", IntegerType())
    .add("Area (km²)", IntegerType())
    .add("Density (per km²)", DoubleType())
    .add("Growth Rate", DoubleType())
    .add("World Population Percentage", DoubleType())
)

def clean_col_name(name):
    return re.sub(r"[^0-9a-zA-Z_]", "_", name).strip("_")

df_bronze = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(population_schema)
    .load(volume_path)
)
for old_name in df_bronze.columns:
    new_name = clean_col_name(old_name)
    if new_name != old_name:
        df_bronze = df_bronze.withColumnRenamed(old_name, new_name)

df_bronze.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.brz_world_population")

response = requests.get("https://countries.dev/countries", timeout=15)
response.raise_for_status()
data = response.json()

df_bronze_api = spark.createDataFrame(pd.DataFrame(data))
df_bronze_api.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.brz_countries_api")

print(f"Bronze OK - population: {df_bronze_pop.count()} rows, API: {df_bronze_api.count()} rows")

##### Silver layer


In [0]:
RENAME_MAP = {
    "Country_Territory": "country",
    "2022_Population": "population_2022",
    "Area__km": "area_km2",
    "Density__per_km": "density_km2",
    "Growth_Rate": "growth_rate",
    "World_Population_Percentage": "world_population_pct"
}

df_population_silver = spark.table(f"{catalog}.{schema}.brz_world_population")
for old_name, new_name in RENAME_MAP.items():
    df_population_silver  = df_population_silver .withColumnRenamed(old_name, new_name)

for field in df_population_silver .schema.fields:
    if isinstance(field.dataType, T.StringType):
        df_population_silver  = df_population_silver.withColumn(field.name, F.trim(F.col(field.name)))

df_population_silver = (
    df_population_silver .select(
        "CCA3", "country", "Capital", "Continent",
        "population_2022", "area_km2", "density_km2",
        "growth_rate", "world_population_pct"
    )
    .dropDuplicates()
    .dropna(subset=["country", "CCA3"])
)

df_api = (
    spark.table(f"{catalog}.{schema}.brz_countries_api")
    .select(
        "alpha3Code", "capital", "region",
        F.col("currencies")[0]["code"].alias("currency_code")
    )
    .dropDuplicates()
    .dropna(subset=["alpha3Code"])
)

df_silver = (
    df_population_silver .alias("p")
    .join(df_api.alias("a"), F.col("p.CCA3") == F.col("a.alpha3Code"), "left")
    .select(
        "p.CCA3", "p.country", "p.Continent",
        "p.population_2022", "p.area_km2", "p.density_km2", "p.growth_rate",
        "a.capital", "a.currency_code", "a.region"
    )
)

df_silver.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.slv_population_enriched")
df_silver.printSchema()
print(f"Filas en Silver: {df_silver.count()}")

##### Gold layer

In [0]:
df_silver = spark.table(f"{catalog}.{schema}.slv_population_enriched")

df_gold = (
    df_silver.groupBy("Continent")
    .agg(
        F.sum("population_2022").alias("total_population"),
        F.avg("density_km2").alias("avg_density"),
        F.count("country").alias("num_countries"),
        F.avg("growth_rate").alias("avg_growth_rate")
    )
    .orderBy(F.desc("total_population"))
)

df_gold.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.gld_population_by_continent")
df_gold.display()